# Phases 3 to 5 — Scheduling results

**Study:** Fair and carbon-aware spatiotemporal scheduling of AI workloads in data centres under forecast uncertainty

This notebook assembles every scheduling result, reproduces the tables that appear in the
manuscript, and runs the consistency checks that guard them.

**Regeneration.** Set `REGENERATE = True` to re-execute the simulations from the cached grid
and trace data rather than reading the stored result files. The full pipeline takes about
nineteen minutes on a single core, of which thirteen are the initial grid retrieval; see
`src/run_all.sh` for the stage-by-stage timings. With `REGENERATE = False` the notebook reads
the stored CSVs, which are the exact files the figures are drawn from.

In [1]:
REGENERATE = False

import os, subprocess, sys, time
import numpy as np, pandas as pd
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd())=="notebooks" else os.getcwd()
OUT = os.path.join(ROOT, "outputs")
T0 = time.time()

def stage(script, *args):
    if REGENERATE:
        t = time.time()
        subprocess.run([sys.executable, os.path.join(ROOT, "src", script), *args], check=True)
        print("%s %s -> %.1f s" % (script, " ".join(args), time.time() - t))

def load(name):
    return pd.read_csv(os.path.join(OUT, name))

print("regenerate:", REGENERATE)

regenerate: False


## 1. Policy comparison on the design period

Months 1 to 24 only. Policy parameters are selected here and frozen before the held-out
period is touched.

In [2]:
stage("simulate_policies.py", "policies")
p = load("phase3c_policies.csv")
tbl = p.pivot_table(index="B_h", columns="policy", values="saving_pct").round(2)
tbl = tbl[["threshold", "percentile", "lyapunov", "horizon", "oracle"]]
display(tbl)

policy,threshold,percentile,lyapunov,horizon,oracle
B_h,,,,,
4,2.71,1.90,2.71,3.88,5.34
12,4.80,3.48,5.03,6.71,8.29
24,5.76,3.81,6.48,7.85,9.18


### Standing assertions

Three checks are enforced in the simulator and repeated here, because each caught a real
defect during development: work conservation, dominance of perfect foresight over any
forecast-based policy, and monotonicity of the saving in the deferral budget.

In [3]:
issues = []
for B, g in p.groupby("B_h"):
    orc = float(g.loc[g.policy == "oracle", "saving_pct"].iloc[0])
    for pol in ("threshold", "percentile", "horizon", "lyapunov"):
        v = float(g.loc[g.policy == pol, "saving_pct"].iloc[0])
        if v > orc + 0.05:
            issues.append("A2: %s beats perfect foresight at B=%s" % (pol, B))
for pol in ("horizon", "oracle"):
    s = p[p.policy == pol].sort_values("B_h").saving_pct.to_numpy()
    if np.any(np.diff(s) < -0.05):
        issues.append("A3: %s not monotone in budget: %s" % (pol, np.round(s, 2)))
print("A2 oracle dominance and A3 budget monotonicity:", "PASS" if not issues else issues)

A2 oracle dominance and A3 budget monotonicity: PASS


## 2. Held-out evaluation

Months 25 to 36, scored once with every parameter frozen. Each policy yields twelve monthly
savings against the carbon-agnostic baseline; the interval is a percentile bootstrap over
those months and significance uses a Wilcoxon signed rank test corrected by Holm-Bonferroni
within each signal and budget.

In [4]:
stage("evaluate_held_out.py")
h = load("phase5_held_out.csv")
main = h[(h.signal == "S1_published")][["B_h","policy","mean_saving_pct","ci_low","ci_high",
                                        "months_positive","n_months","p_holm","pct_of_oracle"]]
display(main.round(3).reset_index(drop=True))
print("all comparisons significant at 5%% after correction:", bool((h.p_holm < 0.05).all()))

,B_h,policy,mean_saving_pct,ci_low,ci_high,months_positive,n_months,p_holm,pct_of_oracle
0,4,threshold,3.925,2.881,5.049,12,12,0.001,59.085
1,4,percentile,2.727,1.751,3.780,12,12,0.001,41.052
2,4,horizon,5.111,4.073,6.162,12,12,0.001,76.947
3,4,lyapunov,3.920,2.863,5.017,12,12,0.001,59.012
4,4,oracle,6.642,5.423,7.817,12,12,0.001,100.000
5,12,threshold,7.151,4.621,9.683,12,12,0.001,69.296
6,12,percentile,4.954,2.444,7.490,9,12,0.003,48.012
7,12,horizon,8.896,6.599,11.131,12,12,0.001,86.208
8,12,lyapunov,7.434,5.157,9.766,12,12,0.001,72.038
9,12,oracle,10.319,8.024,12.585,12,12,0.001,100.000


all comparisons significant at 5%% after correction: True


### Both signals

S1 is the published operational forecast. S2 is the day-ahead forecast built in Phase 3e from
information available at least 24 hours ahead. A claim enters the manuscript only where it
holds under both.

In [5]:
both = h[h.policy.isin(["horizon","oracle"])].pivot_table(
    index=["B_h","policy"], columns="signal", values="mean_saving_pct").round(2)
display(both)
hz = h[h.policy=="horizon"].pivot_table(index="B_h", columns="signal", values="pct_of_oracle").round(1)
print("\nShare of achievable saving realised (%):"); display(hz)
print("Under the published forecast the share rises with the budget; under the day-ahead")
print("forecast it falls. Forecast quality and scheduling flexibility substitute at the margin.")

signal       S1_published  S2_day_ahead
B_h policy                             
4   horizon          5.11          5.34
    oracle           6.64          6.64
12  horizon          8.90          7.73
    oracle          10.32         10.32
24  horizon         10.06          7.89
    oracle          11.28         11.28


Share of achievable saving realised (%):


signal,S1_published,S2_day_ahead
B_h,,
4,76.9,80.3
12,86.2,74.9
24,89.2,69.9


Under the published forecast the share rises with the budget; under the day-ahead
forecast it falls. Forecast quality and scheduling flexibility substitute at the margin.


## 3. Sensitivities

Three sweeps establish what the saving actually depends on: the power model, the available
headroom, and the deferral budgets granted to tenants.

In [6]:
stage("simulate_policies.py", "energy"); stage("simulate_policies.py", "utilisation")
stage("simulate_policies.py", "flexibility"); stage("simulate_policies.py", "fairness")
e, u = load("phase3d_energy_bracket.csv"), load("phase3c_utilisation.csv")
f, fr = load("phase3c_flexibility.csv"), load("phase3c_fairness.csv")
print("Energy bracket — facility saving by idle-to-peak ratio")
display(e.pivot_table(index="idle_fraction", columns="B_h",
                      values=["saving_pct","facility_saving_pct"]).round(2))
print("Headroom — saving by mean utilisation"); display(u[["utilisation","saving_pct","deadline_violation_pct"]].round(2))
print("Flexibility grant"); display(f[["grant_h","saving_pct","tenant_jain","tenant_spread_gCO2_kWh","worst_tenant_intensity"]].round(3))
print("Allocation lever (negative result)"); display(fr[["fair_lambda","saving_pct","tenant_jain","tenant_spread_gCO2_kWh"]].round(5))

Energy bracket — facility saving by idle-to-peak ratio


facility_saving_pct       saving_pct      
B_h                            12    24         12    24
idle_fraction                                           
0.1                          3.68  4.31       6.71  7.85
0.3                          2.75  3.22       6.71  7.85
0.6                          1.48  1.73       6.71  7.85

Headroom — saving by mean utilisation


,utilisation,saving_pct,deadline_violation_pct
0,0.60,12.83,0.00
1,0.74,9.95,0.00
2,0.82,6.71,0.00
3,0.90,4.64,0.01
4,0.98,1.04,0.67


Flexibility grant


,grant_h,saving_pct,tenant_jain,tenant_spread_gCO2_kWh,worst_tenant_intensity
0,0.0,7.853,0.999,8.064,125.949
1,8.0,8.591,1.000,0.611,118.822
2,12.0,8.837,1.000,1.049,118.811
3,24.0,9.567,1.000,0.556,117.489


Allocation lever (negative result)


,fair_lambda,saving_pct,tenant_jain,tenant_spread_gCO2_kWh
0,0.0,7.85255,0.99944,8.06391
1,1.0,7.85255,0.99944,8.05630
2,5.0,7.85255,0.99944,8.04348
3,20.0,7.85255,0.99944,8.04812


The allocation sweep is a negative result and is reported as one. Reweighting who receives
capacity within a period changes nothing, because a tenant's achievable intensities are fixed
by its deadline before any allocation decision is made. Granting slack changes both the saving
and the equity outcome, and changes them in the same direction.

## 4. Spatial placement

Scored on the forecast field, because no regional outturn is published. The temporal baseline
is scored on the same field so the comparison is internally consistent, and the figures are
bounds rather than measured savings.

In [7]:
stage("simulate_spatial.py", "12", "0.85"); stage("simulate_spatial.py", "3", "0.45")
hi, lo = load("phase4_spatial_u85.csv"), load("phase4_spatial_u45.csv")
print("Busy fleet, utilisation 0.85, twelve months")
display(hi[["mode","exposure_cap","saving_pct","top_region_share_pct","herfindahl"]].round(3))
print("Spare capacity, utilisation 0.45, three months")
display(lo[["mode","exposure_cap","saving_pct","top_region_share_pct","herfindahl"]].round(3))
st = load("phase4_spatial_stress_u85.csv")
print("\nStress test, decisions on the forecast scored against a perturbed field:")
display(st.groupby(["mode"], dropna=False).saving_pct.agg(["mean","std"]).round(3))

Busy fleet, utilisation 0.85, twelve months


,mode,exposure_cap,saving_pct,top_region_share_pct,herfindahl
0,agnostic,NaN,0.000,7.143,0.071
1,temporal,NaN,13.326,8.435,0.072
2,spatial,NaN,22.350,10.675,0.080
3,spatiotemporal,NaN,34.654,9.667,0.084
4,spatiotemporal,2.0,34.632,9.661,0.084
5,spatiotemporal,1.5,34.599,9.657,0.084
6,spatiotemporal,1.1,16.297,7.857,0.074


Spare capacity, utilisation 0.45, three months


,mode,exposure_cap,saving_pct,top_region_share_pct,herfindahl
0,agnostic,NaN,0.000,7.143,0.071
1,temporal,NaN,28.075,12.263,0.075
2,spatial,NaN,58.920,17.098,0.128
3,spatiotemporal,NaN,71.753,20.178,0.164
4,spatiotemporal,2.0,58.439,14.283,0.119
5,spatiotemporal,1.5,44.006,10.716,0.094
6,spatiotemporal,1.1,19.557,7.859,0.074



Stress test, decisions on the forecast scored against a perturbed field:


,mean,std
mode,,
spatiotemporal,34.295,0.068
temporal,13.137,0.030


## 5. Figures

Every figure is drawn from the stored CSVs above, never from a value typed into the plotting
script. The manifest records which file each panel draws on.

In [8]:
stage("make_figures.py")
display(load("figure_manifest.csv"))
print("wall-clock for this notebook: %.1f s" % (time.time() - T0))

,figure,sources,content
0,fig1_signal_accuracy,phase3e_signal_accuracy.csv,MAE by signal and split; held-out correlation ...
1,fig2_held_out_saving,phase5_held_out.csv,mean saving with bootstrap intervals and share...
2,fig3_policy_comparison,phase5_held_out.csv,"held-out saving by policy and budget, publishe..."
3,fig4_energy_and_headroom,phase3d_energy_bracket.csv; phase3c_utilisatio...,facility saving against idle fraction; saving ...
4,fig5_flexibility_grant,phase3c_flexibility.csv,saving and tenant intensity spread against gra...
5,fig6_spatial_and_exposure,phase4_spatial_u85.csv; phase4_spatial_u45.csv,saving by placement mode; saving against cumul...


wall-clock for this notebook: 0.2 s
